# Split-Only Oscillation → BFS Merge

**Motivation.** Min-sum belief propagation with the split-only policy (each factor cloned into two halves, no damping) almost always falls into a *period-2 oscillation* on random factor graphs. Two consecutive iterations after the tail kicks in differ only in their assignment vectors — these are the two oscillating *branches*.

**What this notebook does.**
1. Build a single random factor graph (large enough and noisy enough that split-only oscillates).
2. Run min-sum split-only BP and detect period-2 oscillation.
3. Grab two consecutive snapshots once oscillation is established → `branch1` and `branch2`.
4. Do **one** BFS over the variable-only projection from a fixed root. The BFS order is shared by both branches; only the per-node assignment differs.
5. Print the two branches side-by-side in BFS order.
6. Score each branch (**cost BEFORE merge**) with the **original** (pre-split) cost tables.
7. Apply the **alternating BFS merge** (at every disagreement node, alternate `b1, b2, b1, …`) to produce two merged candidates, then report **cost AFTER merge** and the before/after delta.
8. Visualize the factor graph and the cost-vs-iteration oscillation.

In [ ]:
from __future__ import annotations

import sys
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from propflow import (
    FGBuilder,
    SplitEngine,
    DampingEngine,
    MinSumComputator,
    CTFactories,
)
from propflow.bp.engines import DampingSCFGEngine
from propflow.core.agents import VariableAgent, FactorAgent

# Make experiments/other/non_convergence_chain/code importable for the oscillation detector.
_REPO_ROOT = Path.cwd()
while _REPO_ROOT != _REPO_ROOT.parent and not (_REPO_ROOT / "experiments").is_dir():
    _REPO_ROOT = _REPO_ROOT.parent
_DETECTOR_DIR = _REPO_ROOT / "experiments" / "other" / "non_convergence_chain" / "code"
if str(_DETECTOR_DIR) not in sys.path:
    sys.path.insert(0, str(_DETECTOR_DIR))
from oscillation_detector import (  # type: ignore  # noqa: E402
    classification_details,
    detect_even_odd_oscillation,
    detect_period,
)

## 1. Build the random factor graph

Larger graphs with high-range integer costs reliably oscillate under split-only min-sum. If the chosen seed yields convergence or a non-period-2 regime, change `SEED` and re-run.

In [ ]:
SEED = 7
NUM_VARS = 20
DOMAIN_SIZE = 5
DENSITY = 0.25
COST_LOW, COST_HIGH = 100, 200
MAX_ITER = 1000

np.random.seed(SEED)

fg = FGBuilder.build_random_graph(
    num_vars=NUM_VARS,
    domain_size=DOMAIN_SIZE,
    ct_factory=CTFactories.RANDOM_INT,
    ct_params={"low": COST_LOW, "high": COST_HIGH},
    density=DENSITY,
    seed=SEED,
)
print(f"Built FG: {len(fg.variables)} vars, {len(fg.factors)} factors")

## 2. Snapshot the original cost tables and variable-only projection

`SplitEngine.post_init()` mutates the graph in place by replacing every factor with two split clones. We need the original tables and topology *before* that happens, both for scoring assignments later and for BFS / visualization.

In [ ]:
ORIG_TABLES: dict[str, np.ndarray] = {f.name: f.cost_table.copy() for f in fg.factors}
ORIG_FACTOR_VARS: dict[str, list[str]] = {
    f.name: [v.name for v in fg.edges[f]] for f in fg.factors
}
VAR_NAMES: list[str] = sorted(v.name for v in fg.variables)

# Variable-only projection: two variables are connected iff they share a factor.
Gvar = nx.Graph()
Gvar.add_nodes_from(VAR_NAMES)
for fname, vs in ORIG_FACTOR_VARS.items():
    for i in range(len(vs)):
        for j in range(i + 1, len(vs)):
            Gvar.add_edge(vs[i], vs[j], factor=fname)

# Bipartite (variables + factors) for the topology plot, rebuilt from saved
# metadata so the figure is independent of fg's later in-place splits.
Gbip = nx.Graph()
Gbip.add_nodes_from(((v, {"bipartite": 0}) for v in VAR_NAMES))
for fname, vs in ORIG_FACTOR_VARS.items():
    Gbip.add_node(fname, bipartite=1)
    for v in vs:
        Gbip.add_edge(fname, v)

print(f"Variable-only graph: {Gvar.number_of_nodes()} nodes, {Gvar.number_of_edges()} edges")
print(f"Components: {nx.number_connected_components(Gvar)}")

## 3. Run split-only min-sum

In [ ]:
engine = SplitEngine(
    factor_graph=fg,
    computator=MinSumComputator(),
    split_factor=0.5,
)
engine.run(max_iter=MAX_ITER)
print(f"Captured {len(engine._snapshots)} snapshots")

## 4. Detect period-2 oscillation and pick the two branches

In [ ]:
snapshots = [engine._snapshots[i] for i in sorted(engine._snapshots)]
trace = [
    {
        "assignments": dict(snap.assignments),
        "beliefs": {k: np.asarray(v) for k, v in snap.beliefs.items()},
    }
    for snap in snapshots
]

details = classification_details(trace)
even_odd = detect_even_odd_oscillation(trace)
period_info = detect_period(trace, max_period=2, min_repeats=3)

print("Classification details:")
for k, v in details.items():
    print(f"  {k}: {v}")
print()
print(f"Even/odd oscillation: {even_odd}")

if not even_odd["is_even_odd"]:
    raise RuntimeError(
        "No period-2 oscillation detected for this seed. "
        "Try a different SEED, change DENSITY, or increase NUM_VARS / cost range."
    )

tail_start = period_info["start"] if period_info else max(0, len(snapshots) - 2)
# Use the last full period in the tail so we are deep in the oscillation regime.
k = len(snapshots) - 2
branch1 = dict(snapshots[k].assignments)
branch2 = dict(snapshots[k + 1].assignments)
print(f"\nTail starts at iter {tail_start}; using snapshots {k} and {k + 1} as branch1/branch2.")

## 5. BFS over the variable-only graph

One BFS, fixed root — the same node ordering for both branches. Disconnected components are appended deterministically so every variable appears in the output. We also record the BFS parent of each node, used by the greedy merge below.

In [ ]:
ROOT = sorted(Gvar.nodes(), key=lambda v: (int(v[1:]) if v[1:].isdigit() else 0))[0]

def bfs_walk(graph: nx.Graph, root: str) -> tuple[list[str], list[tuple[str, str]], dict[str, str | None]]:
    order: list[str] = []
    tree_edges: list[tuple[str, str]] = []
    parent: dict[str, str | None] = {}
    seen: set[str] = set()

    def bfs_from(start: str) -> None:
        order.append(start)
        seen.add(start)
        parent[start] = None
        for u, v in nx.bfs_edges(graph, source=start):
            if v not in seen:
                seen.add(v)
                order.append(v)
                tree_edges.append((u, v))
                parent[v] = u

    bfs_from(root)
    remaining_components = [
        comp for comp in nx.connected_components(graph) if not (comp & seen)
    ]
    for comp in sorted(remaining_components, key=lambda c: sorted(c)[0]):
        comp_root = sorted(comp, key=lambda v: (int(v[1:]) if v[1:].isdigit() else 0))[0]
        bfs_from(comp_root)
    return order, tree_edges, parent

bfs_order, bfs_tree_edges, bfs_parent = bfs_walk(Gvar, ROOT)
print(f"BFS root: {ROOT}")
print(f"BFS order ({len(bfs_order)} nodes): {bfs_order}")
assert set(bfs_order) == set(VAR_NAMES), "BFS must visit every variable"
assert len(bfs_order) == len(VAR_NAMES), "BFS order must be unique"

## 6. Side-by-side branch comparison in BFS order

`=` marks variables where the two branches agree (those nodes contribute the same value to both merged candidates). `≠` marks the variables that genuinely oscillate.

In [ ]:
rows = []
for v in bfs_order:
    a, b = branch1[v], branch2[v]
    rows.append({"variable": v, "branch1": a, "branch2": b, "match": "=" if a == b else "≠"})
branches_df = pd.DataFrame(rows)
n_diff = int((branches_df["match"] == "≠").sum())
print(f"{n_diff}/{len(branches_df)} variables differ between the two branches")
branches_df

## 7. Score each branch against the original cost tables (BEFORE merge)

The split policy distributes each original cost table across two clones (`p·C`, `(1−p)·C`). To compare assignments in the **original problem**, we re-index the saved pre-split tables.

In [ ]:
def score_assignment(
    assignment: dict[str, int],
    tables: dict[str, np.ndarray],
    factor_vars: dict[str, list[str]],
) -> float:
    total = 0.0
    for fname, table in tables.items():
        idx = tuple(int(assignment[v]) for v in factor_vars[fname])
        total += float(table[idx])
    return total

score_branch1 = score_assignment(branch1, ORIG_TABLES, ORIG_FACTOR_VARS)
score_branch2 = score_assignment(branch2, ORIG_TABLES, ORIG_FACTOR_VARS)
print(f"score(branch1) = {score_branch1:.2f}")
print(f"score(branch2) = {score_branch2:.2f}")
print(f"best of the two = {min(score_branch1, score_branch2):.2f}")

## 8. Alternating BFS merge (cost AFTER merge)

Walk the BFS in order. For each variable `v`:

- If `branch1[v] == branch2[v]` (the two branches **agree** at `v`): take that shared value — no choice to make.
- If they **disagree**: pick from `branch1` the first time, from `branch2` the next time, alternating on every subsequent disagreement.

This produces two merged candidates:

- `merged_1` starts the alternation with `branch1` → at disagreement nodes it picks `b1, b2, b1, b2, …`
- `merged_2` starts the alternation with `branch2` → it picks `b2, b1, b2, b1, …`

So at every oscillating variable the two merged candidates take **opposite** values, and every "agreement" variable is shared by both. This is a constructive way to mix the two oscillating branches in BFS order.

In [ ]:
def alternating_merge(start_branch: str) -> dict[str, int]:
    """Walk BFS order. At agreement nodes use the shared value; at disagreement
    nodes alternate b1/b2/b1/... starting from ``start_branch``.
    """
    if start_branch not in ("branch1", "branch2"):
        raise ValueError("start_branch must be 'branch1' or 'branch2'")
    take = start_branch
    merged: dict[str, int] = {}
    for v in bfs_order:
        if branch1[v] == branch2[v]:
            merged[v] = int(branch1[v])
            continue
        merged[v] = int(branch1[v]) if take == "branch1" else int(branch2[v])
        take = "branch2" if take == "branch1" else "branch1"
    return merged

merged_1 = alternating_merge("branch1")  # b1, b2, b1, ... at disagreement nodes
merged_2 = alternating_merge("branch2")  # b2, b1, b2, ... at disagreement nodes

score_merged_1 = score_assignment(merged_1, ORIG_TABLES, ORIG_FACTOR_VARS)
score_merged_2 = score_assignment(merged_2, ORIG_TABLES, ORIG_FACTOR_VARS)

summary_df = pd.DataFrame(
    [
        {"candidate": "branch1",  "phase": "BEFORE merge", "cost": score_branch1},
        {"candidate": "branch2",  "phase": "BEFORE merge", "cost": score_branch2},
        {"candidate": "merged_1", "phase": "AFTER merge",  "cost": score_merged_1},
        {"candidate": "merged_2", "phase": "AFTER merge",  "cost": score_merged_2},
    ]
)

best_before = min(score_branch1, score_branch2)
best_after = min(score_merged_1, score_merged_2)
delta = best_after - best_before
improvement = "better" if delta < 0 else ("worse" if delta > 0 else "tied")
print(f"Best BEFORE merge: {best_before:.2f}")
print(f"Best AFTER  merge: {best_after:.2f}")
print(f"Δ (after − before): {delta:+.2f}  ({improvement})")
summary_df

In [ ]:
# Per-variable side-by-side: branch1 | branch2 | merged_1 | merged_2 in BFS order.
rows = []
for v in bfs_order:
    b1, b2, m1, m2 = branch1[v], branch2[v], merged_1[v], merged_2[v]
    rows.append({
        "variable": v,
        "branch1": b1,
        "branch2": b2,
        "merged_1": m1,
        "merged_2": m2,
        "m1_picked": "b1" if m1 == b1 else ("b2" if m1 == b2 else "?"),
        "m2_picked": "b1" if m2 == b1 else ("b2" if m2 == b2 else "?"),
    })
merge_df = pd.DataFrame(rows)
n_m1_flipped = int(((merge_df["branch1"] != merge_df["branch2"]) & (merge_df["m1_picked"] == "b2")).sum())
n_m2_flipped = int(((merge_df["branch1"] != merge_df["branch2"]) & (merge_df["m2_picked"] == "b1")).sum())
print(f"merged_1 flipped to branch2 at {n_m1_flipped} of {n_diff} oscillating nodes")
print(f"merged_2 flipped to branch1 at {n_m2_flipped} of {n_diff} oscillating nodes")
merge_df

## 9. Optimal solution via branch-and-bound

20 vars × domain 5 is 5²⁰ ≈ 10¹⁴ combinations — too many to enumerate, but small enough for branch-and-bound with a tight upper-bound warm-start.

**Bound.** For a partial assignment, the lower bound is `sum over factors of (exact cost if all of f's vars are assigned, else min over the still-unassigned axes given the fixed ones)`. Maintaining per-factor contributions incrementally and only recomputing the factors touching the newly-assigned variable keeps the per-node cost small.

**Warm start.** Initialize the incumbent upper bound from `min(score_branch1, score_branch2, score_merged_1, score_merged_2)` — the merge already gave us 7430 — so pruning bites immediately. Variable order is descending degree in the variable-only graph.

In [ ]:
import hashlib
import json
import time


def branch_and_bound(
    variables: list[str],
    factor_vars: dict[str, list[str]],
    tables: dict[str, np.ndarray],
    domain_size: int,
    initial_upper_bound: float,
    initial_assignment: dict[str, int] | None = None,
    time_limit_s: float | None = 300.0,
) -> tuple[float, dict[str, int], dict]:
    """Branch-and-bound search for the assignment minimizing total cost."""
    # Variable order: descending degree in the variable-only projection.
    deg: dict[str, int] = {v: 0 for v in variables}
    for vs in factor_vars.values():
        for u in vs:
            for w in vs:
                if u != w:
                    deg[u] += 1
    var_order = sorted(variables, key=lambda v: (-deg[v], v))

    factors_touching: dict[str, list[str]] = {v: [] for v in variables}
    for fname, vs in factor_vars.items():
        for v in vs:
            factors_touching[v].append(fname)

    factor_contrib: dict[str, float] = {f: float(tables[f].min()) for f in factor_vars}
    running_bound = sum(factor_contrib.values())

    best_cost = float(initial_upper_bound)
    best_assignment: dict[str, int] = dict(initial_assignment) if initial_assignment else {}
    assignment: dict[str, int] = {}

    start = time.time()
    stats = {"nodes": 0, "prunes": 0, "complete": False, "elapsed_s": 0.0}

    def factor_min_given_partial(fname: str) -> float:
        vs = factor_vars[fname]
        tbl = tables[fname]
        if all(v in assignment for v in vs):
            return float(tbl[tuple(assignment[v] for v in vs)])
        slicer = tuple(
            assignment[v] if v in assignment else slice(None) for v in vs
        )
        return float(tbl[slicer].min())

    def search(depth: int, bound_in: float) -> None:
        nonlocal best_cost, best_assignment
        stats["nodes"] += 1

        if time_limit_s is not None and (time.time() - start) > time_limit_s:
            raise TimeoutError()

        if depth == len(var_order):
            if bound_in < best_cost:
                best_cost = bound_in
                best_assignment = dict(assignment)
            return

        v = var_order[depth]
        touched = factors_touching[v]
        saved = {f: factor_contrib[f] for f in touched}

        for val in range(domain_size):
            assignment[v] = val
            delta = 0.0
            for f in touched:
                new = factor_min_given_partial(f)
                delta += new - factor_contrib[f]
                factor_contrib[f] = new
            new_bound = bound_in + delta
            if new_bound < best_cost:
                search(depth + 1, new_bound)
            else:
                stats["prunes"] += 1
            for f in touched:
                factor_contrib[f] = saved[f]
        del assignment[v]

    try:
        search(0, running_bound)
        stats["complete"] = True
    except TimeoutError:
        stats["complete"] = False

    stats["elapsed_s"] = time.time() - start
    return best_cost, best_assignment, stats


def _bb_cache_key() -> str:
    """Hash the inputs that fully determine the B&B optimum."""
    h = hashlib.sha256()
    h.update(
        f"{NUM_VARS}|{DOMAIN_SIZE}|{DENSITY}|{COST_LOW}|{COST_HIGH}|{SEED}".encode()
    )
    for fname in sorted(ORIG_FACTOR_VARS):
        h.update(fname.encode())
        h.update("|".join(ORIG_FACTOR_VARS[fname]).encode())
        h.update(np.ascontiguousarray(ORIG_TABLES[fname]).tobytes())
    return h.hexdigest()[:16]


CACHE_DIR = _REPO_ROOT / "notebooks" / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
_BB_CACHE_PATH = CACHE_DIR / f"bb_optimal_{_bb_cache_key()}.json"

bb_warm = min(score_branch1, score_branch2, score_merged_1, score_merged_2)
bb_warm_assignment = min(
    [
        (score_branch1, branch1),
        (score_branch2, branch2),
        (score_merged_1, merged_1),
        (score_merged_2, merged_2),
    ],
    key=lambda x: x[0],
)[1]

if _BB_CACHE_PATH.exists():
    with _BB_CACHE_PATH.open() as _f:
        _cached = json.load(_f)
    opt_cost = float(_cached["opt_cost"])
    opt_assignment = {k: int(v) for k, v in _cached["opt_assignment"].items()}
    bb_stats = _cached["bb_stats"]
    print(f"Loaded B&B optimum from cache: {_BB_CACHE_PATH.name}")
    print(f"OPTIMAL cost = {opt_cost:.2f}  "
          f"(originally: {bb_stats['nodes']:,} nodes, "
          f"{bb_stats['prunes']:,} prunes, {bb_stats['elapsed_s']:.2f} s)")
else:
    print(f"No cache at {_BB_CACHE_PATH.name}; running B&B from scratch.")
    print(f"B&B starting with warm upper bound = {bb_warm:.2f}")
    opt_cost, opt_assignment, bb_stats = branch_and_bound(
        variables=VAR_NAMES,
        factor_vars=ORIG_FACTOR_VARS,
        tables=ORIG_TABLES,
        domain_size=DOMAIN_SIZE,
        initial_upper_bound=bb_warm,
        initial_assignment=bb_warm_assignment,
        time_limit_s=300.0,
    )
    status = "OPTIMAL (search exhausted)" if bb_stats["complete"] else "BEST KNOWN (timed out)"
    print(f"\n{status}: cost = {opt_cost:.2f}")
    print(f"B&B stats: {bb_stats['nodes']:,} nodes, {bb_stats['prunes']:,} prunes, "
          f"{bb_stats['elapsed_s']:.2f} s")
    # Only cache when the search was exhausted — partial timeouts shouldn't masquerade as optimal.
    if bb_stats["complete"]:
        with _BB_CACHE_PATH.open("w") as _f:
            json.dump(
                {
                    "opt_cost": opt_cost,
                    "opt_assignment": opt_assignment,
                    "bb_stats": bb_stats,
                    "params": {
                        "SEED": SEED, "NUM_VARS": NUM_VARS, "DOMAIN_SIZE": DOMAIN_SIZE,
                        "DENSITY": DENSITY, "COST_LOW": COST_LOW, "COST_HIGH": COST_HIGH,
                    },
                },
                _f,
                indent=2,
            )
        print(f"Saved B&B optimum to cache: {_BB_CACHE_PATH.name}")

In [ ]:
# Compare all candidates against the optimum.
final_rows = [
    {"candidate": "branch1",  "phase": "BEFORE merge", "cost": score_branch1},
    {"candidate": "branch2",  "phase": "BEFORE merge", "cost": score_branch2},
    {"candidate": "merged_1", "phase": "AFTER merge",  "cost": score_merged_1},
    {"candidate": "merged_2", "phase": "AFTER merge",  "cost": score_merged_2},
    {"candidate": "OPTIMAL" if bb_stats["complete"] else "BEST KNOWN",
     "phase": "B&B", "cost": opt_cost},
]
final_df = pd.DataFrame(final_rows)
final_df["gap_to_opt"] = final_df["cost"] - opt_cost
final_df["pct_gap"] = (final_df["gap_to_opt"] / opt_cost) * 100.0
print(final_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

merged_matches_opt = (
    score_merged_1 == opt_cost or score_merged_2 == opt_cost
)
print()
if merged_matches_opt:
    print("✓ At least one merged candidate is OPTIMAL.")
else:
    best_merged = min(score_merged_1, score_merged_2)
    print(f"Best merged = {best_merged:.2f}, optimum = {opt_cost:.2f} "
          f"(gap {best_merged - opt_cost:.2f}, {(best_merged - opt_cost) / opt_cost * 100:.2f}%)")

## 10. Damping run — compare to the best split-only branch

Run min-sum on a freshly-built identical FG, this time with **Q-damping** (`damping_factor=0.9`). Damping is the canonical fix for the period-2 oscillation we just observed. We compare its final assignment to the **best of the two oscillating branches** (the lower-cost of `branch1`/`branch2`, **before** any merge) side-by-side in BFS order.

In [ ]:
DAMPING_FACTOR = 0.9

# Rebuild the same FG: identical topology (seed) and identical cost tables
# (numpy RNG re-seeded to SEED). We verify table-equality below before trusting it.
np.random.seed(SEED)
fg_damp = FGBuilder.build_random_graph(
    num_vars=NUM_VARS,
    domain_size=DOMAIN_SIZE,
    ct_factory=CTFactories.RANDOM_INT,
    ct_params={"low": COST_LOW, "high": COST_HIGH},
    density=DENSITY,
    seed=SEED,
)
for f in fg_damp.factors:
    assert np.array_equal(f.cost_table, ORIG_TABLES[f.name]), \
        f"FG rebuild mismatch at factor {f.name}; SEED/numpy state is not deterministic."

damp_engine = DampingEngine(
    factor_graph=fg_damp,
    computator=MinSumComputator(),
    damping_factor=DAMPING_FACTOR,
)
damp_engine.run(max_iter=MAX_ITER)
damp_snapshots = [damp_engine._snapshots[i] for i in sorted(damp_engine._snapshots)]

# Diagnose damping behavior on this trace (did it actually converge?).
damp_trace = [
    {
        "assignments": dict(snap.assignments),
        "beliefs": {k: np.asarray(v) for k, v in snap.beliefs.items()},
    }
    for snap in damp_snapshots
]
damp_details = classification_details(damp_trace)
print(f"Damping classification: {damp_details['classification']}, "
      f"period={damp_details['period']}, "
      f"assignment_converged={damp_details['assignment_converged']}")

damp_final = dict(damp_snapshots[-1].assignments)
score_damp = score_assignment(damp_final, ORIG_TABLES, ORIG_FACTOR_VARS)

# Pick the best of the two raw (pre-merge) split-only branches.
if score_branch1 <= score_branch2:
    best_branch_name, best_branch_score, best_branch_assignment = "branch1", score_branch1, branch1
else:
    best_branch_name, best_branch_score, best_branch_assignment = "branch2", score_branch2, branch2

print(f"\nBest split-only raw branch: {best_branch_name} (cost {best_branch_score:.2f})")
print(f"Damping final assignment cost (damping={DAMPING_FACTOR}): {score_damp:.2f}")
print(f"Δ (damping − best_raw_branch): {score_damp - best_branch_score:+.2f}")
print(f"Δ (damping − OPTIMAL):         {score_damp - opt_cost:+.2f}")

In [ ]:
# Side-by-side: best-raw-split-branch vs damping, in BFS order.
rows = []
for v in bfs_order:
    bb = int(best_branch_assignment[v])
    dd = int(damp_final[v])
    rows.append({
        "variable": v,
        best_branch_name: bb,
        f"damping (λ={DAMPING_FACTOR})": dd,
        "OPTIMAL": int(opt_assignment[v]),
        "match": "=" if bb == dd else "≠",
        "matches_OPT": "=" if dd == int(opt_assignment[v]) else "≠",
    })
damp_vs_branch_df = pd.DataFrame(rows)
n_diff_db = int((damp_vs_branch_df["match"] == "≠").sum())
n_damp_opt = int((damp_vs_branch_df["matches_OPT"] == "=").sum())
print(f"{n_diff_db}/{len(damp_vs_branch_df)} variables differ between "
      f"{best_branch_name} and damping")
print(f"{n_damp_opt}/{len(damp_vs_branch_df)} variables in damping match the OPTIMAL assignment")
damp_vs_branch_df

## 11. Splitting + damping (`DampingSCFGEngine`)

Combine factor splitting with Q-damping (`split_factor=0.5`, `damping_factor=0.9`) on the same FG. This is the canonical fix in the codebase for the period-2 oscillation we see with split-only. We score it on the **original** cost tables and compare against the raw branches, the alternating merge, plain damping, and the optimum.

In [ ]:
SPLIT_DAMP_FACTOR = 0.5
SPLIT_DAMP_DAMPING = 0.9

# Fresh FG with identical topology and cost tables.
np.random.seed(SEED)
fg_sd = FGBuilder.build_random_graph(
    num_vars=NUM_VARS, domain_size=DOMAIN_SIZE,
    ct_factory=CTFactories.RANDOM_INT,
    ct_params={"low": COST_LOW, "high": COST_HIGH},
    density=DENSITY, seed=SEED,
)
for f in fg_sd.factors:
    assert np.array_equal(f.cost_table, ORIG_TABLES[f.name])

sd_engine = DampingSCFGEngine(
    factor_graph=fg_sd,
    computator=MinSumComputator(),
    split_factor=SPLIT_DAMP_FACTOR,
    damping_factor=SPLIT_DAMP_DAMPING,
)
sd_engine.run(max_iter=MAX_ITER)
sd_snapshots = [sd_engine._snapshots[i] for i in sorted(sd_engine._snapshots)]
sd_trace = [
    {"assignments": dict(s.assignments),
     "beliefs": {k: np.asarray(v) for k, v in s.beliefs.items()}}
    for s in sd_snapshots
]
sd_details = classification_details(sd_trace)
print(f"Split+Damping classification: {sd_details['classification']}, "
      f"period={sd_details['period']}, "
      f"assignment_converged={sd_details['assignment_converged']}")

sd_final = dict(sd_snapshots[-1].assignments)
score_sd = score_assignment(sd_final, ORIG_TABLES, ORIG_FACTOR_VARS)
print(f"\nscore(split+damping) = {score_sd:.2f}")
print(f"  vs best raw branch ({best_branch_name}): {score_sd - best_branch_score:+.2f}")
print(f"  vs best merged candidate:               {score_sd - min(score_merged_1, score_merged_2):+.2f}")
print(f"  vs plain damping:                       {score_sd - score_damp:+.2f}")
print(f"  vs OPTIMAL:                             {score_sd - opt_cost:+.2f}")

In [ ]:
# Master summary table covering every candidate against OPTIMAL.
master_rows = [
    {"candidate": "branch1",                "phase": "BEFORE merge",      "cost": score_branch1},
    {"candidate": "branch2",                "phase": "BEFORE merge",      "cost": score_branch2},
    {"candidate": "merged_1",               "phase": "AFTER merge",       "cost": score_merged_1},
    {"candidate": "merged_2",               "phase": "AFTER merge",       "cost": score_merged_2},
    {"candidate": f"damping (λ={DAMPING_FACTOR})",
     "phase": "damping",            "cost": score_damp},
    {"candidate": f"split+damping (p={SPLIT_DAMP_FACTOR}, λ={SPLIT_DAMP_DAMPING})",
     "phase": "split+damping",      "cost": score_sd},
    {"candidate": "OPTIMAL" if bb_stats["complete"] else "BEST KNOWN",
     "phase": "B&B",                "cost": opt_cost},
]
master_df = pd.DataFrame(master_rows)
master_df["gap_to_opt"] = master_df["cost"] - opt_cost
master_df["pct_gap"] = (master_df["gap_to_opt"] / opt_cost) * 100.0
print(master_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

In [ ]:
# Unified BFS table:
#   branches keep the b1/b2 agreement indicator;
#   merged candidates keep the m1_picked / m2_picked indicator;
#   OPTIMAL and split+damping show just the chosen domain value;
#   the last three columns flag whether split+damping / OPTIMAL agree with one of
#   the raw branches, and whether OPTIMAL == split+damping.
rows = []
for v in bfs_order:
    b1, b2 = int(branch1[v]), int(branch2[v])
    m1, m2 = int(merged_1[v]), int(merged_2[v])
    opt = int(opt_assignment[v])
    sdv = int(sd_final[v])
    rows.append({
        "variable": v,
        "branch1": b1,
        "branch2": b2,
        "match": "=" if b1 == b2 else "≠",
        "merged_1": m1,
        "merged_2": m2,
        "m1_picked": "b1" if m1 == b1 else ("b2" if m1 == b2 else "?"),
        "m2_picked": "b1" if m2 == b1 else ("b2" if m2 == b2 else "?"),
        "OPTIMAL": opt,
        "split+damping": sdv,
        "sd_in_branches":  "=" if sdv in (b1, b2) else "≠",
        "opt_in_branches": "=" if opt in (b1, b2) else "≠",
        "opt_=_sd":        "=" if opt == sdv else "≠",
    })
unified_bfs_df = pd.DataFrame(rows)
unified_bfs_df

In [ ]:
# Flip split+damping at every variable where OPTIMAL agrees with one of the raw
# branches but disagrees with split+damping. Then re-score the resulting assignment.
flip_vars = [
    v for v in bfs_order
    if int(opt_assignment[v]) in (int(branch1[v]), int(branch2[v]))
    and int(opt_assignment[v]) != int(sd_final[v])
]
print(f"Variables to flip ({len(flip_vars)}): {flip_vars}")

sd_flipped = {v: int(sd_final[v]) for v in bfs_order}
for v in flip_vars:
    sd_flipped[v] = int(opt_assignment[v])

flipped_series = pd.Series(
    [sd_flipped[v] for v in bfs_order],
    index=bfs_order,
    name="sd_flipped",
)
print("\nFull BFS-ordered flipped assignment:")
print(flipped_series.to_string())

score_sd_flipped = score_assignment(sd_flipped, ORIG_TABLES, ORIG_FACTOR_VARS)
print(
    f"\nscore(sd_flipped)    = {score_sd_flipped:.2f}"
    f"\nscore(split+damping) = {score_sd:.2f}  (Δ {score_sd_flipped - score_sd:+.2f})"
    f"\nOPTIMAL              = {opt_cost:.2f}  (gap {score_sd_flipped - opt_cost:+.2f}, "
    f"{(score_sd_flipped - opt_cost) / opt_cost * 100:.2f}%)"
)

In [ ]:
# Simple split-only BP cost trace with the two chosen branches highlighted (single seed).
_costs = np.array([snap.global_cost for snap in snapshots], dtype=float)
_iters = np.arange(len(_costs))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(_iters, _costs, color="#2E7D32", lw=1.2,
        label="split-only BP cost (split tables)")
ax.scatter([k, k + 1], [_costs[k], _costs[k + 1]], s=80,
           color=["#E63946", "#1D3557"], zorder=5,
           label=f"branch1 (iter {k}) / branch2 (iter {k + 1})")
ax.axvline(tail_start, ls="--", color="gray", alpha=0.7,
           label=f"tail start (iter {tail_start})")
ax.set_xlabel("BP iteration")
ax.set_ylabel("global cost (split tables)")
ax.set_title(f"Split-only oscillation (seed={SEED}) — period={details['period']}")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## 12bis. Seed sweep — split-only, split+damping, and the new sd-aware merge

For each of 30 seeds, build the FG, run split-only (capture the two oscillating branches) and split+damping (capture its final assignment), then apply a **new merge rule that uses all three** (branch1, branch2, split+damping):

- if `branch1[v] == branch2[v]` (branches agree): both merged candidates take `sd[v]`;
- elif `sd[v] ∈ {branch1[v], branch2[v]}` (sd already lies on one branch): both merged candidates take `sd[v]`;
- else (branches disagree **and** sd is outside both): merge_1 alternates `b1, b2, b1, …` across such variables in BFS order; merge_2 alternates `b2, b1, b2, …`.

No branch-and-bound here — just per-seed scores on the original cost tables.

In [ ]:
SWEEP_SEEDS = list(range(30))


def _bfs_order_from_factors(factor_vars: dict[str, list[str]],
                            var_names: list[str]) -> list[str]:
    Gv = nx.Graph()
    Gv.add_nodes_from(var_names)
    for vs in factor_vars.values():
        for i in range(len(vs)):
            for j in range(i + 1, len(vs)):
                Gv.add_edge(vs[i], vs[j])
    if not var_names:
        return []
    root = sorted(var_names, key=lambda v: (int(v[1:]) if v[1:].isdigit() else 0))[0]
    order = [root]
    seen = {root}
    for _, vv in nx.bfs_edges(Gv, source=root):
        if vv not in seen:
            order.append(vv); seen.add(vv)
    # Cover any remaining disconnected components deterministically.
    for comp in sorted(nx.connected_components(Gv), key=lambda c: sorted(c)[0]):
        if comp & seen:
            continue
        comp_root = sorted(comp, key=lambda v: (int(v[1:]) if v[1:].isdigit() else 0))[0]
        order.append(comp_root); seen.add(comp_root)
        for _, vv in nx.bfs_edges(Gv, source=comp_root):
            if vv not in seen:
                order.append(vv); seen.add(vv)
    return order


def _new_merge(b1: dict[str, int], b2: dict[str, int], sd: dict[str, int],
               order: list[str]) -> tuple[dict[str, int], dict[str, int]]:
    m1, m2 = {}, {}
    take1, take2 = "b1", "b2"
    for v in order:
        bv1, bv2, sv = int(b1[v]), int(b2[v]), int(sd[v])
        if bv1 == bv2:
            m1[v] = m2[v] = sv
        elif sv == bv1 or sv == bv2:
            m1[v] = m2[v] = sv
        else:
            m1[v] = bv1 if take1 == "b1" else bv2
            m2[v] = bv1 if take2 == "b1" else bv2
            take1 = "b2" if take1 == "b1" else "b1"
            take2 = "b2" if take2 == "b1" else "b1"
    return m1, m2


def _run_seed(seed: int) -> dict:
    # Build FG twice (once consumed by SplitEngine, once by DampingSCFGEngine).
    np.random.seed(seed)
    fg_a = FGBuilder.build_random_graph(
        num_vars=NUM_VARS, domain_size=DOMAIN_SIZE,
        ct_factory=CTFactories.RANDOM_INT,
        ct_params={"low": COST_LOW, "high": COST_HIGH},
        density=DENSITY, seed=seed,
    )
    tables = {f.name: f.cost_table.copy() for f in fg_a.factors}
    factor_vars = {f.name: [v.name for v in fg_a.edges[f]] for f in fg_a.factors}
    var_names = sorted(v.name for v in fg_a.variables)
    order = _bfs_order_from_factors(factor_vars, var_names)

    sp = SplitEngine(factor_graph=fg_a, computator=MinSumComputator(), split_factor=0.5)
    sp.run(max_iter=MAX_ITER)
    sp_snaps = [sp._snapshots[i] for i in sorted(sp._snapshots)]
    b1 = dict(sp_snaps[-2].assignments)
    b2 = dict(sp_snaps[-1].assignments)

    np.random.seed(seed)
    fg_b = FGBuilder.build_random_graph(
        num_vars=NUM_VARS, domain_size=DOMAIN_SIZE,
        ct_factory=CTFactories.RANDOM_INT,
        ct_params={"low": COST_LOW, "high": COST_HIGH},
        density=DENSITY, seed=seed,
    )
    sde = DampingSCFGEngine(
        factor_graph=fg_b, computator=MinSumComputator(),
        split_factor=SPLIT_DAMP_FACTOR, damping_factor=SPLIT_DAMP_DAMPING,
    )
    sde.run(max_iter=MAX_ITER)
    sd_snaps = [sde._snapshots[i] for i in sorted(sde._snapshots)]
    sd = dict(sd_snaps[-1].assignments)

    m1, m2 = _new_merge(b1, b2, sd, order)

    sd_class = classification_details([
        {"assignments": dict(s.assignments),
         "beliefs": {k: np.asarray(v) for k, v in s.beliefs.items()}}
        for s in sd_snaps
    ])
    sp_class = classification_details([
        {"assignments": dict(s.assignments),
         "beliefs": {k: np.asarray(v) for k, v in s.beliefs.items()}}
        for s in sp_snaps
    ])

    # mismatches: variables where sd[v] is in NEITHER branch1[v] nor branch2[v]
    mismatches = [v for v in order if int(sd[v]) not in (int(b1[v]), int(b2[v]))]

    return {
        "seed": seed,
        "branch1": score_assignment(b1, tables, factor_vars),
        "branch2": score_assignment(b2, tables, factor_vars),
        "split+damping": score_assignment(sd, tables, factor_vars),
        "merge_1": score_assignment(m1, tables, factor_vars),
        "merge_2": score_assignment(m2, tables, factor_vars),
        "sp_class": sp_class["classification"],
        "sd_class": sd_class["classification"],
        "n_vars": len(order),
        "mismatches_sd_vs_branches": len(mismatches),
    }


_sweep_t0 = time.time()
sweep_rows = []
for _s in SWEEP_SEEDS:
    _r = _run_seed(_s)
    sweep_rows.append(_r)
    print(f"seed={_s:>2}  b1={_r['branch1']:>6.0f}  b2={_r['branch2']:>6.0f}  "
          f"sd={_r['split+damping']:>6.0f}  m1={_r['merge_1']:>6.0f}  m2={_r['merge_2']:>6.0f}  "
          f"mismatches={_r['mismatches_sd_vs_branches']:>2}/{_r['n_vars']}")
print(f"\nSweep took {time.time() - _sweep_t0:.1f} s")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df["best_branch"] = sweep_df[["branch1", "branch2"]].min(axis=1)
sweep_df["best_merge"]  = sweep_df[["merge_1",  "merge_2"]].min(axis=1)
sweep_df["best_overall_heur"] = sweep_df[["best_branch", "split+damping", "best_merge"]].min(axis=1)
sweep_df["merge_beats_sd"]   = sweep_df["best_merge"] < sweep_df["split+damping"]
sweep_df["merge_beats_branch"] = sweep_df["best_merge"] < sweep_df["best_branch"]
sweep_df["sd_beats_branch"]    = sweep_df["split+damping"] < sweep_df["best_branch"]

In [ ]:
# Full per-seed table (all candidate scores + classifications + mismatch count).
pd.set_option("display.max_columns", None)
print("Full per-seed sweep:")
print(sweep_df.to_string(index=False, float_format=lambda x: f"{x:.0f}"))

print("\nAggregate stats over the sweep:")
print(f"  mean best_branch       = {sweep_df['best_branch'].mean():.1f}")
print(f"  mean split+damping     = {sweep_df['split+damping'].mean():.1f}")
print(f"  mean best_merge        = {sweep_df['best_merge'].mean():.1f}")
print(f"  mean best_overall      = {sweep_df['best_overall_heur'].mean():.1f}")
print(f"  best_merge beats sd         : {sweep_df['merge_beats_sd'].sum()}/{len(sweep_df)}")
print(f"  best_merge beats best_branch: {sweep_df['merge_beats_branch'].sum()}/{len(sweep_df)}")
print(f"  sd beats best_branch        : {sweep_df['sd_beats_branch'].sum()}/{len(sweep_df)}")
print(f"  mean mismatches sd vs branches: "
      f"{sweep_df['mismatches_sd_vs_branches'].mean():.2f} / {sweep_df['n_vars'].iloc[0]}")

In [ ]:
# Focused final table: seed | branch1_score | branch2_score | mismatches (sd vs branches).
focus_df = sweep_df[["seed", "branch1", "branch2", "mismatches_sd_vs_branches"]].rename(
    columns={
        "branch1": "branch1_score",
        "branch2": "branch2_score",
        "mismatches_sd_vs_branches": "mismatches_sd_vs_b1b2",
    }
)
focus_df

## 12. Visualize the factor graph and the BFS tree

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (a) Bipartite factor graph
ax = axes[0]
var_nodes = [n for n, d in Gbip.nodes(data=True) if d["bipartite"] == 0]
fac_nodes = [n for n, d in Gbip.nodes(data=True) if d["bipartite"] == 1]
pos_bip = nx.spring_layout(Gbip, seed=SEED)
nx.draw_networkx_edges(Gbip, pos_bip, ax=ax, alpha=0.4)
nx.draw_networkx_nodes(Gbip, pos_bip, nodelist=var_nodes, node_color="#4C9BE8",
                       node_size=420, ax=ax, label="variables")
nx.draw_networkx_nodes(Gbip, pos_bip, nodelist=fac_nodes, node_color="#F2A65A",
                       node_shape="s", node_size=180, ax=ax, label="factors")
nx.draw_networkx_labels(Gbip, pos_bip, labels={v: v for v in var_nodes}, font_size=9, ax=ax)
ax.set_title("Bipartite factor graph")
ax.legend(scatterpoints=1, loc="upper right")
ax.axis("off")

# (b) Variable-only projection with BFS tree highlighted
ax = axes[1]
pos_var = nx.spring_layout(Gvar, seed=SEED)
non_tree_edges = [e for e in Gvar.edges() if e not in set(bfs_tree_edges)
                  and (e[1], e[0]) not in set(bfs_tree_edges)]
nx.draw_networkx_edges(Gvar, pos_var, edgelist=non_tree_edges, ax=ax,
                       alpha=0.25, style="dashed")
nx.draw_networkx_edges(Gvar, pos_var, edgelist=bfs_tree_edges, ax=ax,
                       width=2.2, edge_color="#2E7D32")
differs = {v for v in bfs_order if branch1[v] != branch2[v]}
agree_nodes = [v for v in bfs_order if v not in differs]
nx.draw_networkx_nodes(Gvar, pos_var, nodelist=agree_nodes, node_color="#BDBDBD",
                       node_size=380, ax=ax, label="branch1 == branch2")
nx.draw_networkx_nodes(Gvar, pos_var, nodelist=list(differs), node_color="#E63946",
                       node_size=420, ax=ax, label="oscillating")
nx.draw_networkx_nodes(Gvar, pos_var, nodelist=[ROOT], node_color="#FFD166",
                       node_size=620, edgecolors="black", linewidths=2, ax=ax,
                       label="BFS root")
nx.draw_networkx_labels(Gvar, pos_var, font_size=9, ax=ax)
ax.set_title("Variable-only graph (BFS tree in green)")
ax.legend(scatterpoints=1, loc="upper right")
ax.axis("off")

plt.tight_layout()
plt.show()

## 13. Cost-vs-iteration: split-only vs damping vs split+damping

All three engines run for `MAX_ITER = 1000` iterations.

- **Left axis**: per-iteration BP global cost from each engine on **its own** tables (split-only sees split tables; damping sees original tables; split+damping sees split tables).
- **Right axis**: assignment scores on the **original** cost tables — comparable across engines. The black line is the OPTIMAL from B&B.

In [ ]:
costs       = np.array([snap.global_cost for snap in snapshots],       dtype=float)
damp_costs  = np.array([snap.global_cost for snap in damp_snapshots],  dtype=float)
sd_costs    = np.array([snap.global_cost for snap in sd_snapshots],    dtype=float)

iters       = np.arange(len(costs))
damp_iters  = np.arange(len(damp_costs))
sd_iters    = np.arange(len(sd_costs))

fig, ax = plt.subplots(figsize=(12, 4.8))

# Left axis: per-iteration BP global cost (each engine on its own tables).
ax.plot(iters, costs, color="#2E7D32", lw=1.3,
        label="split-only BP cost (split tables)")
ax.plot(damp_iters, damp_costs, color="#118AB2", lw=1.3, ls="--",
        label=f"damping BP cost (λ={DAMPING_FACTOR}, original tables)")
ax.plot(sd_iters, sd_costs, color="#9C27B0", lw=1.3, ls="-.",
        label=f"split+damping BP cost (p={SPLIT_DAMP_FACTOR}, λ={SPLIT_DAMP_DAMPING}, split tables)")
ax.scatter([k, k + 1], [costs[k], costs[k + 1]], s=80,
           color=["#E63946", "#1D3557"], zorder=5,
           label=f"branch1 (iter {k}) / branch2 (iter {k + 1})")
ax.axvline(tail_start, ls="--", color="gray", alpha=0.6,
           label=f"split-only tail start (iter {tail_start})")
ax.set_xlabel("BP iteration")
ax.set_ylabel("global cost (per-engine tables)")

# Right axis: scored assignments on the ORIGINAL cost tables.
ax2 = ax.twinx()
ax2.axhline(score_branch1,  color="#E63946", ls=":", alpha=0.7,
            label=f"branch1       = {score_branch1:.0f}")
ax2.axhline(score_branch2,  color="#1D3557", ls=":", alpha=0.7,
            label=f"branch2       = {score_branch2:.0f}")
ax2.axhline(score_merged_1, color="#E63946", ls="-", lw=1.0, alpha=0.55,
            label=f"merged_1      = {score_merged_1:.0f}")
ax2.axhline(score_merged_2, color="#1D3557", ls="-", lw=1.0, alpha=0.55,
            label=f"merged_2      = {score_merged_2:.0f}")
ax2.axhline(score_damp, color="#F2A65A", ls="-", lw=1.5, alpha=0.9,
            label=f"damping       = {score_damp:.0f}")
ax2.axhline(score_sd,   color="#9C27B0", ls="-", lw=1.5, alpha=0.9,
            label=f"split+damping = {score_sd:.0f}")
opt_label = "OPTIMAL" if bb_stats["complete"] else "best known"
ax2.axhline(opt_cost, color="#000000", ls="-", lw=2.0, alpha=0.9,
            label=f"{opt_label}       = {opt_cost:.0f}")
ax2.set_ylabel("original-table cost (scored assignments)")

ax.set_title(
    f"split-only vs damping vs split+damping (seed={SEED}, max_iter={MAX_ITER}) — "
    f"split-only={details['classification']}, "
    f"damping={damp_details['classification']}, "
    f"split+damping={sd_details['classification']}"
)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2,
          loc="center left", bbox_to_anchor=(1.08, 0.5),
          fontsize=8, ncol=1, frameon=False)
plt.tight_layout()
plt.show()

## 14. Final assignment table: OPTIMAL vs split+damping (BFS order)

In [ ]:
final_df = pd.DataFrame(
    [
        {
            "variable": v,
            "OPTIMAL": int(opt_assignment[v]),
            "split+damping": int(sd_final[v]),
        }
        for v in bfs_order
    ]
)
final_df

## Future work

- **Multi-root BFS**: run the merge from every variable as the root and deduplicate the resulting merged candidates. Different roots yield different BFS orders and therefore different alternation patterns at disagreement nodes.
- **Seed sweep**: sweep `SEED` across hundreds of random instances and report (a) the rate of period-2 vs other classifications, (b) the typical gap between `score(branch1)` and `score(branch2)`, and (c) how often the alternating merge strictly beats both raw branches.
- **Greedy / exhaustive merge variants**: instead of strict alternation, try (i) a greedy per-node pick that locally minimizes cost given already-assigned BFS-ancestors, or (ii) full enumeration of `2^k` per-node parity choices over the `k` oscillating variables. Compare against the alternating baseline.